## Configurando o MySQL com Python

Neste momento, vamos configurar a integração entre o MySQL e Python. Isso nos permitirá executar consultas, inserções e outras operações no banco de dados MySQL usando o Python como nossa linguagem de programação principal.

In [1]:
import mysql.connector

cnx = mysql.connector.connect(
    host="localhost",
    user="dennis_farias",
    password="134679",
    auth_plugin='mysql_native_password'
)

print("Conectado!", cnx)


Conectado! <mysql.connector.connection.MySQLConnection object at 0x7f0a82d6a170>


Criaremos um cursor, que é um objeto utilizado para executar instruções SQL no contexto do Python. Essse cursor nos permitirá enviar consultas e comandos SQL para o banco de dados MySQL por meio da conexão estabelecida e obter os resultados de volta para serem processados em nosso código Python

In [2]:
cursor = cnx.cursor()

## Criando uma base de dados

O método `execute()` é usado para compilar uma instrução SQL

In [3]:
cursor.execute("CREATE DATABASE IF NOT EXISTS dbprodutos;")

In [4]:
cursor.execute("SHOW DATABASES;")
for db in cursor:
    print(db)

(bytearray(b'dbprodutos'),)
(bytearray(b'information_schema'),)
(bytearray(b'mysql'),)
(bytearray(b'performance_schema'),)
(bytearray(b'sys'),)


## Criando uma tabela

Antes de criar nossa tabela, vamos revisar as colunas necessárias que precisaremos incluir nela. Essa etapa é importante para garantir que nossa tabela seja projetada corretamente, atendendo aos requisitos de armazenamento e organização dos dados.

In [5]:
import pandas as pd

df_livros = pd.read_csv('/home/dennis/pipeline-python-mongo-mysql/data/tabela_livros.csv')
df_livros.head()

,_id,Produto,Categoria do Produto,Preço,Frete,Data da Compra,Vendedor,Local da compra,Avaliação da compra,Tipo de pagamento,Quantidade de parcelas,Latitude,Longitude
0,691bc5e0f587c3e50ae4f33a,Modelagem preditiva,livros,92.45,5.609697,2020-01-01,Thiago Silva,BA,1,cartao_credito,3,-13.29,-41.71
1,691bc5e0f587c3e50ae4f391,Modelagem preditiva,livros,71.64,1.399054,2023-03-01,Mariana Ferreira,SP,4,cartao_credito,2,-22.19,-48.79
2,691bc5e0f587c3e50ae4f3b8,Ciência de dados com python,livros,58.61,4.060654,2021-05-01,Beatriz Moraes,PR,4,cartao_credito,1,-24.89,-51.55
3,691bc5e0f587c3e50ae4f3cb,Dashboards com Power BI,livros,55.44,3.639025,2021-06-01,Camila Ribeiro,SP,3,cartao_credito,10,-22.19,-48.79
4,691bc5e0f587c3e50ae4f3e4,Dashboards com Power BI,livros,46.87,0.000000,2021-07-01,Beatriz Moraes,RS,5,cartao_credito,8,-30.17,-53.50


In [6]:
df_livros.columns

Index(['_id', 'Produto', 'Categoria do Produto', 'Preço', 'Frete',
       'Data da Compra', 'Vendedor', 'Local da compra', 'Avaliação da compra',
       'Tipo de pagamento', 'Quantidade de parcelas', 'Latitude', 'Longitude'],
      dtype='object')

In [7]:
df_livros.shape

(742, 13)

In [8]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS dbprodutos.tb_livros(
               id VARCHAR(100),
               Produto VARCHAR(100),
               Categoria_Produto VARCHAR(100),
               Preco FLOAT(10,2),
               Frete FLOAT(10,2),
               Data_Compra DATE,
               Vendedor VARCHAR(100),
               Local_Compra VARCHAR(100),
               Avaliacao_Compra INT,
               Tipo_Pagamento VARCHAR(100),
               Qntd_Parcelas INT,
               Latitude FLOAT(10,2),
               Longitude FLOAT(10,2),

               PRIMARY KEY (id)
               );
""")

**Selecionando a base de dados para verificar a tabela criada**

In [9]:
cursor.execute("USE dbprodutos;")
cursor.execute("SHOW TABLES;")

for tb in cursor:
    print(tb)

(bytearray(b'tb_livros'),)


## Inserindo os dados do csv na tabela

Para inserir os dados na tabela do MySQL, é necessário percorrer cada linha do DataFrame e transformá-las em tuplas. Essa abordagem permite que mapeemos os dados do DataFrame para as colunas correspondentes da tabela do MySQL de forma eficiente e precisa.

In [ ]:
for i, row in df_livros.iterrows():
    print(tuple(row))

In [ ]:
lista_dados = [tuple(row) for i, row in df_livros.iterrows()]
lista_dados

In [12]:
sql = "INSERT INTO dbprodutos.tb_livros VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s);"

cursor.executemany(sql, lista_dados)
cnx.commit()

In [13]:
print(cursor.rowcount, "dados inseridos")

742 dados inseridos


In [14]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS dbprodutos.tb_produtos_2021_em_diante(
               id VARCHAR(100),
               Produto VARCHAR(100),
               Categoria_Produto VARCHAR(100),
               Preco FLOAT(10,2),
               Frete FLOAT(10,2),
               Data_Compra DATE,
               Vendedor VARCHAR(100),
               Local_Compra VARCHAR(100),
               Avaliacao_Compra INT,
               Tipo_Pagamento VARCHAR(100),
               Qntd_Parcelas INT,
               Latitude FLOAT(10,2),
               Longitude FLOAT(10,2),

               PRIMARY KEY (id));
""")

In [15]:
import pandas as pd
df_produtos = pd.read_csv("../data/tabela_2021_em_diante.csv")
df_produtos.head()

,_id,Produto,Categoria do Produto,Preço,Frete,Data da Compra,Vendedor,Local da compra,Avaliação da compra,Tipo de pagamento,Quantidade de parcelas,Latitude,Longitude
0,691bc5e0f587c3e50ae4f343,Xadrez de madeira,brinquedos,25.23,0.000000,2021-01-01,Thiago Silva,BA,5,cartao_credito,2,-13.29,-41.71
1,691bc5e0f587c3e50ae4f344,Impressora,eletronicos,322.04,14.732100,2021-01-01,João Souza,SP,3,cartao_credito,1,-22.19,-48.79
2,691bc5e0f587c3e50ae4f34c,Blocos de montar,brinquedos,36.84,0.000000,2022-01-01,Pedro Gomes,SP,4,boleto,1,-22.19,-48.79
3,691bc5e0f587c3e50ae4f34e,Fogão,eletrodomesticos,607.49,33.235430,2022-01-01,Juliana Costa,MG,5,cartao_credito,3,-18.10,-44.38
4,691bc5e0f587c3e50ae4f368,Bicicleta,esporte e lazer,520.28,27.986471,2022-02-01,Rafael Costa,SP,5,cartao_credito,5,-22.19,-48.79


In [16]:
df_produtos.shape

(6574, 13)

In [17]:
lista_dados = [tuple(row) for i, row in df_produtos.iterrows()]
sql = "INSERT INTO dbprodutos.tb_produtos_2021_em_diante VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)"

cursor.executemany(sql, lista_dados)
cnx.commit()

In [18]:
print(cursor.rowcount)

6574


## Visualizando os dados inseridos

In [ ]:
cursor.execute("SELECT * FROM dbprodutos.tb_livros;")

for row in cursor:
    print(row)

In [20]:
cursor.close()

True

In [21]:
cnx.close()